# Debug: Humanitarian Text-Only — Compare Our Code vs Reference

Runs the same humanitarian text-only test using:
1. **Approach A**: 100% verbatim reference notebook code
2. **Approach B**: Our `scripts/zeroshot_llama.py`

Then compares results per-sample to find where they diverge.

## Setup

In [1]:
import torch, json, os, re, time, warnings
import pandas as pd
from PIL import Image
from transformers import MllamaForConditionalGeneration, AutoProcessor
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

MODEL_ID = "meta-llama/Llama-3.2-11B-Vision-Instruct"

model = MllamaForConditionalGeneration.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=torch.bfloat16
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model.eval()
print(f"Model loaded on {model.device}")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

Model loaded on cuda:0


## Load Data (shared)

In [2]:
# Load from our preprocessed TSV
our_df = pd.read_csv("../data/CrisisMMD/tasks/humanitarian/text_only/test.tsv", sep="\t")
print(f"Our TSV: {len(our_df)} rows, labels: {our_df['class_label'].value_counts().to_dict()}")
print(f"Columns: {list(our_df.columns)}")
print(f"First text: {our_df.iloc[0]['tweet_text'][:80]}...")

Our TSV: 955 rows, labels: {'not_humanitarian': 504, 'other_relevant_information': 235, 'rescue_volunteering_or_donation_effort': 126, 'infrastructure_and_utility_damage': 81, 'affected_individuals': 9}
Columns: ['tweet_id', 'tweet_text', 'class_label']
First text: .@Lendio has a great event tomorrow for both #BYU and #Utah fans to support Hurr...


## Approach A: 100% verbatim reference notebook code

All functions and prompts copied directly from `temp/llama-3-fewshot-Humanitarian.ipynb`.

In [3]:
# --- 100% verbatim from temp/llama-3-fewshot-Humanitarian.ipynb ---

HUMA_TEXT_ONLY_PROMPT = (
    "You are an expert in disaster response and humanitarian aid data analysis. "
    "Examine this text delimited by triple quotes (\"\"\" \"\"\") carefully and classify it into exactly one of these categories (0-4). "
    "Respond with ONLY the number, no other text or explanation.\n\n"
    "Categories:\n"
    "0: HUMAN IMPACT - Must show direct human suffering or hardship:\n"
    "- Deaths, injuries, or missing people\n"
    "- People struggling without basic needs (food, water, shelter)\n"
    "- Displaced or evacuated people\n"
    "- Personal stories of survival or loss\n"
    "- People stranded or waiting for rescue\n"
    
    "1: RESPONSE EFFORTS - Any organized help effort, no matter how small:\n"
    "- Rescue operations and emergency response\n"
    "- Aid collection or distribution activities\n"
    "- Donations of money, supplies, or services\n"
    "- Volunteer work and relief efforts\n"
    "- Medical assistance\n"
    "- Fundraising events for disaster relief\n"
    
    "2: INFRASTRUCTURE DAMAGE - Must describe specific physical destruction:\n"
    "- Destroyed or damaged buildings and homes\n"
    "- Damaged roads, bridges, or transportation systems\n"
    "- Disrupted power lines or water systems\n"
    "- Damaged vehicles or equipment\n"
    "- Before/after comparisons showing destruction\n"
    
    "3: CRISIS UPDATES - Must be specific to the crisis but not fit above categories:\n"
    "- Weather forecasts and disaster warnings\n"
    "- Maps or descriptions of impact areas\n"
    "- Official announcements about the disaster\n"
    "- Statistics and data about crisis impact\n"
    "- Crisis reporting without specific damage/casualties/response\n"
    
    "4: NOT CRISIS-RELATED - Use when no other category clearly fits:\n"
    "- General discussion without crisis specifics\n"
    "- Personal opinions about non-crisis aspects\n"
    "- Promotional or commercial content\n"
    "- Unclear connection to crisis\n"
    "- Content that could apply to non-crisis situations\n\n"
    
    "Important Decision Rules:\n"
    "- If you see ANY mention of help, rescue, or donations \u2192 Pick 1\n"
    "- If you see ANY mention of human casualties, suffering, or displacement but not related to volunteer, rescue, donation... \u2192 Pick 0\n"    
    "- If you see ANY specific physical destruction of properties \u2192 Pick 2\n"
    "- If it\u2019s clearly about the crisis but doesn\u2019t fit 0-2 \u2192 Pick 3\n"
    "- If multiple categories could apply, use the one BEST FITS the text\n"
    "- Only use 4 when you are COMPLETELY SURE no other category fits\n"
    "Answer with just a single digit (0\u20134)."
)

def detect_humanitarian_label(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    if re.search(r"\b(not (provide|give|contain)|4|\*\*4\*\*)\b", text, re.IGNORECASE):
        return 4
    elif (re.search(r"\bpeople\b", text, re.IGNORECASE) and re.search(r"\bcrisis\b", text, re.IGNORECASE)) or \
         re.search(r'(?=.*\banswer|response|number\b)(?=.*\b0\b)', text, re.IGNORECASE) or \
         re.search(r"\b\*\*0\*\*\b", text, re.IGNORECASE):
        return 0
    elif re.search(r"\b(rescue|volunteering|donation|1|\*\*1\*\*)\b", text, re.IGNORECASE):
        return 1
    elif re.search(r"\b(infrastructure|building|utility|vehicle|car|2|\*\*2\*\*)\b", text, re.IGNORECASE):
        return 2
    elif re.search(r"\b(other|3|\*\*3\*\*)\b", text, re.IGNORECASE):
        return 3
    return 4

def classify_raw_text(raw_text, task="Humanitarian"):
    parts = raw_text.split("assistant")
    if len(parts) > 1:
        final_answer = parts[-1].strip()
        if final_answer.startswith("0"): return 0
        elif final_answer.startswith("1"): return 1
        elif final_answer.startswith("2"): return 2
        elif final_answer.startswith("3"): return 3
        elif final_answer.startswith("4"): return 4
        else: return detect_humanitarian_label(final_answer)
    return None

def construct_message_template(few_shot_examples, prompt, test_text=None, test_image=None, use_texts=True, use_images=True, few_shot_num=0):
    messages = []
    user_content = []
    
    if few_shot_num > 0 and few_shot_examples:          
        for index, example in enumerate(few_shot_examples):
            txt_prompt = "Below is an example for your reference:\n"
            if use_texts and use_images:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image, and here is the Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text and image is: "
            elif use_texts:
                txt_prompt = f"Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text is: "
            else:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image. The category of this sample image is: "
            txt_prompt += str(example['label'])
            user_content.append({"type": "text", "text": txt_prompt})    

    txt_prompt = ""
    if use_texts and use_images:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image, and here is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test image and test Text is: "
    elif use_texts:
        txt_prompt = f"{prompt}\nHere is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test Text is: "
    else:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image that you need to classify. The category of this test image is: "

    user_content.append({"type": "text", "text": txt_prompt})    
    messages.append({"role": "user", "content": user_content})             
    return messages

print("Reference functions loaded")
print(f"Prompt length: {len(HUMA_TEXT_ONLY_PROMPT)} chars")
print(f"Prompt ends with: {repr(HUMA_TEXT_ONLY_PROMPT[-30:])}")

Reference functions loaded
Prompt length: 2250 chars
Prompt ends with: 'ith just a single digit (0–4).'


## Run Approach A

In [5]:
# --- Run Approach A: exact reference notebook flow ---
# Map our string labels to reference's integer labels
LABEL_MAP = {
    "affected_individuals": 0,
    "rescue_volunteering_or_donation_effort": 1,
    "infrastructure_and_utility_damage": 2,
    "other_relevant_information": 3,
    "not_humanitarian": 4,
}

y_true_ref, y_pred_ref = [], []
raw_outputs_ref = []
start = time.time()

for i, row in our_df.iterrows():
    text = row["tweet_text"]
    gold = LABEL_MAP[row["class_label"]]
    
    messages = construct_message_template(
        few_shot_examples=None,
        prompt=HUMA_TEXT_ONLY_PROMPT,
        test_text=text,
        test_image=None,
        use_texts=True,
        use_images=False,
        few_shot_num=0
    )
    
    formatted_input = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=formatted_input, return_tensors="pt", truncation=False).to(model.device)
    
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*do_sample.*")
        output = model.generate(
            **inputs, max_new_tokens=20, min_new_tokens=1,
            do_sample=False, num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    
    raw = processor.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    pred = classify_raw_text(raw, task="Humanitarian")
    
    y_true_ref.append(gold)
    y_pred_ref.append(pred if pred is not None else 4)
    raw_outputs_ref.append(raw)
    
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start
        remaining = (len(our_df) - i - 1) / ((i + 1) / elapsed)
        print(f"  [{i+1}/{len(our_df)}] {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")
        torch.cuda.empty_cache()

acc_ref = accuracy_score(y_true_ref, y_pred_ref)
p_ref, r_ref, f1_ref, _ = precision_recall_fscore_support(y_true_ref, y_pred_ref, average="weighted")

print(f"\n=== Approach A: Reference Notebook Code ===")
print(f"  Accuracy:  {acc_ref:.4f}")
print(f"  Precision: {p_ref:.4f}")
print(f"  Recall:    {r_ref:.4f}")
print(f"  F1 Score:  {f1_ref:.4f}")
print(f"  (Reference reported: Accuracy=0.7445, F1=0.7409)")

  [100/955] 19s elapsed, ~166s remaining
  [200/955] 39s elapsed, ~146s remaining
  [300/955] 58s elapsed, ~127s remaining
  [400/955] 77s elapsed, ~107s remaining
  [500/955] 97s elapsed, ~88s remaining
  [600/955] 116s elapsed, ~69s remaining
  [700/955] 135s elapsed, ~49s remaining
  [800/955] 155s elapsed, ~30s remaining
  [900/955] 175s elapsed, ~11s remaining

=== Approach A: Reference Notebook Code ===
  Accuracy:  0.7403
  Precision: 0.7823
  Recall:    0.7403
  F1 Score:  0.7374
  (Reference reported: Accuracy=0.7445, F1=0.7409)


## Run Approach B: Our Code

In [3]:
import sys
sys.path.insert(0, os.path.abspath(".."))

from scripts.zeroshot_llama import run_zeroshot

DATA_ROOT = os.path.abspath("../data")
OUTPUT_DIR = os.path.abspath("../results/zeroshot_debug")

preds_ours, metrics_ours = run_zeroshot(
    task="humanitarian", modality="text_only", split="test",
    model_id=MODEL_ID, data_root=DATA_ROOT, output_dir=OUTPUT_DIR,
    model=model, processor=processor,
)

print(f"\n=== Approach B: Our Code ===")
print(f"  Accuracy:  {metrics_ours['accuracy']:.4f}")
print(f"  Precision: {metrics_ours['weighted_precision']:.4f}")
print(f"  Recall:    {metrics_ours['weighted_recall']:.4f}")
print(f"  F1 Score:  {metrics_ours['weighted_f1']:.4f}")

Loaded 955 samples from D:\Workspace\Cotrain_CrisisMMD\data\CrisisMMD\tasks\humanitarian\text_only\test.tsv
  [50/955] 10s elapsed, ~181s remaining, 5.0 samples/s
  [100/955] 19s elapsed, ~164s remaining, 5.2 samples/s
  [150/955] 29s elapsed, ~153s remaining, 5.3 samples/s
  [200/955] 38s elapsed, ~143s remaining, 5.3 samples/s
  [250/955] 48s elapsed, ~135s remaining, 5.2 samples/s
  [300/955] 57s elapsed, ~125s remaining, 5.2 samples/s
  [350/955] 67s elapsed, ~115s remaining, 5.3 samples/s
  [400/955] 76s elapsed, ~106s remaining, 5.3 samples/s
  [450/955] 87s elapsed, ~97s remaining, 5.2 samples/s
  [500/955] 97s elapsed, ~88s remaining, 5.2 samples/s
  [550/955] 107s elapsed, ~78s remaining, 5.2 samples/s
  [600/955] 116s elapsed, ~69s remaining, 5.2 samples/s
  [650/955] 126s elapsed, ~59s remaining, 5.2 samples/s
  [700/955] 135s elapsed, ~49s remaining, 5.2 samples/s
  [750/955] 145s elapsed, ~40s remaining, 5.2 samples/s
  [800/955] 155s elapsed, ~30s remaining, 5.2 samples/s

## Compare: Prompt Diff + Per-Sample Diff

In [ ]:
# --- Compare prompts character by character ---
from scripts.zeroshot_llama import HUMA_TEXT_ONLY_PROMPT as OUR_PROMPT, build_messages

sample_text = our_df.iloc[0]["tweet_text"]

# Build reference prompt (as construct_message_template does)
ref_final = f"{HUMA_TEXT_ONLY_PROMPT}\nHere is the test Text that you need to classify: \"\"\"{sample_text}\"\"\". The category of this test Text is: "

# Build our prompt
our_messages, _ = build_messages(OUR_PROMPT, "humanitarian", text=sample_text, modality="text_only")
our_final = our_messages[0]["content"][0]["text"]

print("=== PROMPT LENGTH ===")
print(f"  Reference: {len(ref_final)} chars")
print(f"  Ours:      {len(our_final)} chars")
print(f"  Diff:      {len(our_final) - len(ref_final)} chars")

# Find first difference
for i, (a, b) in enumerate(zip(ref_final, our_final)):
    if a != b:
        print(f"\n=== FIRST DIFFERENCE at char {i} ===")
        print(f"  Ref: ...{repr(ref_final[max(0,i-20):i+20])}...")
        print(f"  Our: ...{repr(our_final[max(0,i-20):i+20])}...")
        break
else:
    if len(ref_final) == len(our_final):
        print("\n=== PROMPTS ARE IDENTICAL ===")
    else:
        print(f"\n=== Same prefix, different length ===")
        shorter = min(len(ref_final), len(our_final))
        print(f"  Ref tail: {repr(ref_final[shorter-20:])}")
        print(f"  Our tail: {repr(our_final[shorter-20:])}")

# --- Per-sample comparison ---
ID2LABEL_REF = {0: "affected_individuals", 1: "rescue_volunteering_or_donation_effort",
                2: "infrastructure_and_utility_damage", 3: "other_relevant_information",
                4: "not_humanitarian"}

y_pred_ref_str = [ID2LABEL_REF.get(p, "not_humanitarian") for p in y_pred_ref]
y_pred_ours_str = [p["predicted_label"] for p in preds_ours]

agree = sum(a == b for a, b in zip(y_pred_ref_str, y_pred_ours_str))
print(f"\n=== PER-SAMPLE AGREEMENT ===")
print(f"  Agreement: {agree}/{len(our_df)} ({agree/len(our_df)*100:.1f}%)")
print(f"  Disagreements: {len(our_df) - agree}")

# Show first 15 disagreements
disagree_idx = [i for i, (a, b) in enumerate(zip(y_pred_ref_str, y_pred_ours_str)) if a != b]
if disagree_idx:
    print(f"\n  First 15 disagreements:")
    for idx in disagree_idx[:15]:
        print(f"    [{idx}] gold={our_df.iloc[idx]['class_label']}, ref={y_pred_ref_str[idx]}, ours={y_pred_ours_str[idx]}")

## Approach A: Image-Only (reference code)

In [6]:
# --- All reference functions + HUMA_IMAGE_ONLY_PROMPT (self-contained) ---

HUMA_IMAGE_ONLY_PROMPT = (
    "You are an expert in disaster response and humanitarian aid image analysis. "
    "Examine the first image carefully and classify it into exactly one of these categories (0-4). "
    "Respond with ONLY the number, no other text or explanation.\n\n"
    "Categories:\n"
    "0: HUMAN IMPACT - Must show PEOPLE who are clearly affected by the disaster: injured, displaced, "
    "evacuated, in temporary shelters, or waiting in lines for aid. People must be visibly impacted.\n"
    "\n"
    "1: RESPONSE EFFORTS - Must show active RESCUE operations, aid distribution, medical treatment, "
    "VOLUNTEER work, DONATION, or evacuation in progress. Look for emergency responders, relief workers, or "
    "organized aid activities.\n"
    "\n"
    "2: INFRASTRUCTURE DAMAGE - Must show clear physical damage to buildings, roads, bridges, power lines, "
    "VEHICLES or other structures that was caused by the disaster. The damage should be obvious and significant.\n"
    "\n"
    "3: OTHER CRISIS INFO - Shows verified crisis-related content that doesn't fit above categories: "
    "maps of affected areas, emergency warnings, official updates, or documentation of crisis conditions. "
    "Must have clear connection to the current disaster.\n"
    "\n"
    "4: NOT CRISIS-RELATED - Use this for:\n"
    "- Images where you're unsure if it's related to the crisis\n"
    "- General photos that could be from any time/place\n"
    " \n"
    "- Images without clear crisis impact or response\n"
    "- Stock photos or promotional images\n"
    "- Any image that doesn't definitively fit categories 0-3\n\n"
    "\n"
    "Important:\n"
    "- If there's ANY sign of rescue or donation, pick 1.\n"
    "- If there's ANY sign of damage, pick 2.\n"
    "- If there's ANY sign of obviously distressed or harmed people, pick 0.\n"
    "- If it's definitely about a crisis but you DO NOT see rescue/damage/impacted people, pick 3.\n"
    "- Otherwise, pick 4. Also, when you are not sure which number to pick, pick 4.\n"
    "Answer with just a single digit (0\u20134)."
)

LABEL_MAP = {
    "affected_individuals": 0,
    "rescue_volunteering_or_donation_effort": 1,
    "infrastructure_and_utility_damage": 2,
    "other_relevant_information": 3,
    "not_humanitarian": 4,
}

def load_and_process_image(image_path):
    assert os.path.isfile(image_path), f"File not found: {image_path}"
    return Image.open(image_path).convert("RGB")

def detect_humanitarian_label(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    if re.search(r"\b(not (provide|give|contain)|4|\*\*4\*\*)\b", text, re.IGNORECASE):
        return 4
    elif (re.search(r"\bpeople\b", text, re.IGNORECASE) and re.search(r"\bcrisis\b", text, re.IGNORECASE)) or \
         re.search(r'(?=.*\banswer|response|number\b)(?=.*\b0\b)', text, re.IGNORECASE) or \
         re.search(r"\b\*\*0\*\*\b", text, re.IGNORECASE):
        return 0
    elif re.search(r"\b(rescue|volunteering|donation|1|\*\*1\*\*)\b", text, re.IGNORECASE):
        return 1
    elif re.search(r"\b(infrastructure|building|utility|vehicle|car|2|\*\*2\*\*)\b", text, re.IGNORECASE):
        return 2
    elif re.search(r"\b(other|3|\*\*3\*\*)\b", text, re.IGNORECASE):
        return 3
    return 4

def classify_raw_text(raw_text, task="Humanitarian"):
    parts = raw_text.split("assistant")
    if len(parts) > 1:
        final_answer = parts[-1].strip()
        if final_answer.startswith("0"): return 0
        elif final_answer.startswith("1"): return 1
        elif final_answer.startswith("2"): return 2
        elif final_answer.startswith("3"): return 3
        elif final_answer.startswith("4"): return 4
        else: return detect_humanitarian_label(final_answer)
    return None

def construct_message_template(few_shot_examples, prompt, test_text=None, test_image=None, use_texts=True, use_images=True, few_shot_num=0):
    messages = []
    user_content = []
    if few_shot_num > 0 and few_shot_examples:
        for index, example in enumerate(few_shot_examples):
            if use_texts and use_images:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image, and here is the Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text and image is: "
            elif use_texts:
                txt_prompt = f"Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text is: "
            else:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image. The category of this sample image is: "
            txt_prompt += str(example['label'])
            user_content.append({"type": "text", "text": txt_prompt})
    txt_prompt = ""
    if use_texts and use_images:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image, and here is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test image and test Text is: "
    elif use_texts:
        txt_prompt = f"{prompt}\nHere is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test Text is: "
    else:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image that you need to classify. The category of this test image is: "
    user_content.append({"type": "text", "text": txt_prompt})
    messages.append({"role": "user", "content": user_content})
    return messages

# Load image-only test data
img_df = pd.read_csv("../data/CrisisMMD/tasks/humanitarian/image_only/test.tsv", sep="\t")
print(f"Image-only TSV: {len(img_df)} rows")

y_true_img, y_pred_img = [], []
start = time.time()

for i, row in img_df.iterrows():
    img = load_and_process_image(row["image_path"])
    gold = LABEL_MAP[row["class_label"]]
    
    messages = construct_message_template(
        few_shot_examples=None, prompt=HUMA_IMAGE_ONLY_PROMPT,
        test_text=None, test_image=img,
        use_texts=False, use_images=True, few_shot_num=0
    )
    
    formatted_input = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=formatted_input, images=img, return_tensors="pt", truncation=False).to(model.device)
    
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*do_sample.*")
        output = model.generate(
            **inputs, max_new_tokens=20, min_new_tokens=1,
            do_sample=False, num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    
    raw = processor.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    pred = classify_raw_text(raw, task="Humanitarian")
    
    y_true_img.append(gold)
    y_pred_img.append(pred if pred is not None else 4)
    
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start
        remaining = (len(img_df) - i - 1) / ((i + 1) / elapsed)
        print(f"  [{i+1}/{len(img_df)}] {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")
        torch.cuda.empty_cache()

acc_img = accuracy_score(y_true_img, y_pred_img)
_, _, f1_img, _ = precision_recall_fscore_support(y_true_img, y_pred_img, average="weighted")
print(f"\n=== Approach A: Image-Only (Reference Code) ===")
print(f"  Accuracy:  {acc_img:.4f}")
print(f"  F1 Score:  {f1_img:.4f}")

Image-only TSV: 955 rows
  [100/955] 80s elapsed, ~681s remaining
  [200/955] 162s elapsed, ~613s remaining
  [300/955] 244s elapsed, ~534s remaining
  [400/955] 326s elapsed, ~452s remaining
  [500/955] 408s elapsed, ~371s remaining
  [600/955] 489s elapsed, ~289s remaining
  [700/955] 569s elapsed, ~207s remaining
  [800/955] 651s elapsed, ~126s remaining
  [900/955] 732s elapsed, ~45s remaining

=== Approach A: Image-Only (Reference Code) ===
  Accuracy:  0.7571
  F1 Score:  0.7637


In [7]:
# Cleanup between runs — free all accumulated data
import gc
del y_true_img, y_pred_img, img_df
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB — ready for next run")

GPU memory allocated: 9.8 GB — ready for next run


## Approach A: Text+Image (reference code)

In [4]:
# --- All reference functions + HUMA_TEXT_IMAGE_PROMPT (self-contained) ---

HUMA_TEXT_IMAGE_PROMPT = (
    "You are an expert in disaster response and humanitarian aid data analysis. "
    "Examine the given text and image carefully and classify them into exactly one of these categories (0-4). "
    "Respond with ONLY the number, no other text or explanation.\n\n"
    "Categories:\n"
    "0: HUMAN IMPACT - Must be about PEOPLE who are clearly affected by the disaster: injured, displaced, "
    "evacuated, in temporary shelters, or waiting in lines for aid. People must be visibly impacted.\n"
    "1: RESPONSE EFFORTS - Must be about active RESCUE operations, aid distribution, medical treatment, "
    "VOLUNTEER work, DONATION, or evacuation in progress. Look for emergency responders, relief workers, or "
    "organized aid activities.\n"
    "2: INFRASTRUCTURE DAMAGE - Must be about clear physical damage to buildings, roads, bridges, power lines, "
    "VEHICLES or other structures that was caused by the disaster. The damage should be obvious and significant.\n"
    "3: OTHER CRISIS INFO - Must be about verified crisis-related content that doesn't fit above categories: "
    "maps of affected areas, emergency warnings, official updates, or documentation of crisis conditions. "
    "Must have clear connection to the current disaster.\n"
    "4: NOT CRISIS-RELATED - Use this for:\n"
    "- Images and text where you're unsure if they are related to the crisis\n"
    "- General texts and photos that could be from any time/place\n"
    "- Texts and images without clear crisis impact or response\n"
    "- Texts are not related to a crisis with stock photos or promotional images\n"
    "- Any text and image that doesn't definitively fit categories 0-3\n\n"
    "Important:\n"
    "- If there's ANY sign of rescue or donation, pick 1.\n"
    "- If there's ANY sign of damage, pick 2.\n"
    "- If there's ANY sign of obviously distressed or harmed people, pick 0.\n"
    "- If the text and image are definitely about a crisis but you DO NOT see rescue/damage/impacted people, pick 3.\n"
    "- Otherwise, pick 4. Also, when you are not sure which number to pick, pick 4.\n"
    "Answer with just a single digit (0\u20134)."
)

LABEL_MAP = {
    "affected_individuals": 0,
    "rescue_volunteering_or_donation_effort": 1,
    "infrastructure_and_utility_damage": 2,
    "other_relevant_information": 3,
    "not_humanitarian": 4,
}

def load_and_process_image(image_path):
    assert os.path.isfile(image_path), f"File not found: {image_path}"
    return Image.open(image_path).convert("RGB")

def detect_humanitarian_label(text):
    text = re.sub(r'[^A-Za-z0-9\s]', '', text)
    if re.search(r"\b(not (provide|give|contain)|4|\*\*4\*\*)\b", text, re.IGNORECASE):
        return 4
    elif (re.search(r"\bpeople\b", text, re.IGNORECASE) and re.search(r"\bcrisis\b", text, re.IGNORECASE)) or \
         re.search(r'(?=.*\banswer|response|number\b)(?=.*\b0\b)', text, re.IGNORECASE) or \
         re.search(r"\b\*\*0\*\*\b", text, re.IGNORECASE):
        return 0
    elif re.search(r"\b(rescue|volunteering|donation|1|\*\*1\*\*)\b", text, re.IGNORECASE):
        return 1
    elif re.search(r"\b(infrastructure|building|utility|vehicle|car|2|\*\*2\*\*)\b", text, re.IGNORECASE):
        return 2
    elif re.search(r"\b(other|3|\*\*3\*\*)\b", text, re.IGNORECASE):
        return 3
    return 4

def classify_raw_text(raw_text, task="Humanitarian"):
    parts = raw_text.split("assistant")
    if len(parts) > 1:
        final_answer = parts[-1].strip()
        if final_answer.startswith("0"): return 0
        elif final_answer.startswith("1"): return 1
        elif final_answer.startswith("2"): return 2
        elif final_answer.startswith("3"): return 3
        elif final_answer.startswith("4"): return 4
        else: return detect_humanitarian_label(final_answer)
    return None

def construct_message_template(few_shot_examples, prompt, test_text=None, test_image=None, use_texts=True, use_images=True, few_shot_num=0):
    messages = []
    user_content = []
    if few_shot_num > 0 and few_shot_examples:
        for index, example in enumerate(few_shot_examples):
            if use_texts and use_images:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image, and here is the Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text and image is: "
            elif use_texts:
                txt_prompt = f"Text: \"\"\"{example['text']}\"\"\"\nThe category of this sample text is: "
            else:
                user_content.append({"type": "image", "image": example["image"]})
                txt_prompt = f"Above is the sample image. The category of this sample image is: "
            txt_prompt += str(example['label'])
            user_content.append({"type": "text", "text": txt_prompt})
    txt_prompt = ""
    if use_texts and use_images:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image, and here is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test image and test Text is: "
    elif use_texts:
        txt_prompt = f"{prompt}\nHere is the test Text that you need to classify: \"\"\"{test_text}\"\"\". The category of this test Text is: "
    else:
        user_content.append({"type": "image", "image": test_image})
        txt_prompt = f"{prompt}\nAbove is the test image that you need to classify. The category of this test image is: "
    user_content.append({"type": "text", "text": txt_prompt})
    messages.append({"role": "user", "content": user_content})
    return messages

# Load text+image test data
ti_df = pd.read_csv("../data/CrisisMMD/tasks/humanitarian/text_image/test.tsv", sep="\t")
print(f"Text+Image TSV: {len(ti_df)} rows")

y_true_ti, y_pred_ti = [], []
start = time.time()

for i, row in ti_df.iterrows():
    img = load_and_process_image(row["image_path"])
    text = row["tweet_text"]
    gold = LABEL_MAP[row["class_label"]]
    
    messages = construct_message_template(
        few_shot_examples=None, prompt=HUMA_TEXT_IMAGE_PROMPT,
        test_text=text, test_image=img,
        use_texts=True, use_images=True, few_shot_num=0
    )
    
    formatted_input = processor.apply_chat_template(messages, add_generation_prompt=True)
    inputs = processor(text=formatted_input, images=img, return_tensors="pt", truncation=False).to(model.device)
    
    with torch.no_grad(), warnings.catch_warnings():
        warnings.filterwarnings("ignore", message=".*do_sample.*")
        output = model.generate(
            **inputs, max_new_tokens=20, min_new_tokens=1,
            do_sample=False, num_beams=1,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id
        )
    
    raw = processor.decode(output[0], skip_special_tokens=True, clean_up_tokenization_spaces=True)
    pred = classify_raw_text(raw, task="Humanitarian")
    
    y_true_ti.append(gold)
    y_pred_ti.append(pred if pred is not None else 4)
    
    if (i + 1) % 100 == 0:
        elapsed = time.time() - start
        remaining = (len(ti_df) - i - 1) / ((i + 1) / elapsed)
        print(f"  [{i+1}/{len(ti_df)}] {elapsed:.0f}s elapsed, ~{remaining:.0f}s remaining")
        torch.cuda.empty_cache()

acc_ti = accuracy_score(y_true_ti, y_pred_ti)
_, _, f1_ti, _ = precision_recall_fscore_support(y_true_ti, y_pred_ti, average="weighted")
print(f"\n=== Approach A: Text+Image (Reference Code) ===")
print(f"  Accuracy:  {acc_ti:.4f}")
print(f"  F1 Score:  {f1_ti:.4f}")

Text+Image TSV: 955 rows
  [100/955] 82s elapsed, ~700s remaining
  [200/955] 165s elapsed, ~623s remaining
  [300/955] 250s elapsed, ~545s remaining
  [400/955] 334s elapsed, ~463s remaining
  [500/955] 419s elapsed, ~382s remaining
  [600/955] 504s elapsed, ~298s remaining
  [700/955] 591s elapsed, ~215s remaining
  [800/955] 675s elapsed, ~131s remaining
  [900/955] 758s elapsed, ~46s remaining

=== Approach A: Text+Image (Reference Code) ===
  Accuracy:  0.7424
  F1 Score:  0.7360


## Cleanup

In [ ]:
import gc
del model, processor
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory allocated: {torch.cuda.memory_allocated()/1e9:.1f} GB")